In [0]:
Bronze_Schema_Path = "assesment_pipline.bronze"

In [0]:
dbutils.widgets.text("valid_records_table_name", "")
valid_records_table_name = dbutils.widgets.get("valid_records_table_name")

In [0]:
expected_silver_count = dbutils.jobs.taskValues.get(
    taskKey="Bronze_Creation",
    key="expected_silver_count",
    debugValue=0
)

In [0]:
valied_bronze_df=spark.read.table(f"{Bronze_Schema_Path}.{valid_records_table_name}")
display(valied_bronze_df)

order_id,order_date,customer_id,product_id,product_name,category,shipping_city,quantity,unit_price,discount,order_status,batch_id,source_file,ingestion_timestamp
ORD0001,2026-10-07,C001,P001,Product 2,GROCERY,Delhi,2,375.0,20.0,PLACED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z
ORD0002,2026-10-08,C002,P002,Product 3,BEVERAGES,Coimbatore,3,550.0,40.0,DELIVERED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z
ORD0003,2026-10-09,C003,P003,Product 4,HOUSEHOLD,Bengaluru,4,725.0,60.0,CANCELLED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z
ORD0004,2026-10-10,C004,P004,Product 5,GROCERY,Chennai,5,900.0,0.0,PLACED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z
ORD0005,2026-10-11,C005,P005,Product 6,BEVERAGES,Hyderabad,1,1075.0,20.0,DELIVERED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z
ORD0006,2026-10-12,C006,P006,Product 7,HOUSEHOLD,Pune,2,1250.0,40.0,CANCELLED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z
ORD0007,2026-10-13,C007,P007,Product 8,GROCERY,Mumbai,3,1425.0,60.0,PLACED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z
ORD0008,2026-10-14,C008,P008,Product 1,BEVERAGES,Kochi,4,200.0,0.0,DELIVERED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z
ORD0009,2026-10-15,C009,P009,Product 2,HOUSEHOLD,Delhi,5,375.0,20.0,CANCELLED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z
ORD0010,2026-10-16,C010,P010,Product 3,GROCERY,Coimbatore,1,550.0,40.0,PLACED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z


In [0]:
from pyspark.sql.functions import *
silver_df=valied_bronze_df.withColumn("product_name",trim(col("product_name"))).withColumn("category",upper(trim(col("category")))).withColumn("shipping_city",upper(trim(col("shipping_city")))).withColumn("net_order_amount",((col("quantity")*col("unit_price"))-col("discount")))
display(silver_df)

order_id,order_date,customer_id,product_id,product_name,category,shipping_city,quantity,unit_price,discount,order_status,batch_id,source_file,ingestion_timestamp,net_order_amount
ORD0001,2026-10-07,C001,P001,Product 2,GROCERY,DELHI,2,375.0,20.0,PLACED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z,730.0
ORD0002,2026-10-08,C002,P002,Product 3,BEVERAGES,COIMBATORE,3,550.0,40.0,DELIVERED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z,1610.0
ORD0003,2026-10-09,C003,P003,Product 4,HOUSEHOLD,BENGALURU,4,725.0,60.0,CANCELLED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z,2840.0
ORD0004,2026-10-10,C004,P004,Product 5,GROCERY,CHENNAI,5,900.0,0.0,PLACED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z,4500.0
ORD0005,2026-10-11,C005,P005,Product 6,BEVERAGES,HYDERABAD,1,1075.0,20.0,DELIVERED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z,1055.0
ORD0006,2026-10-12,C006,P006,Product 7,HOUSEHOLD,PUNE,2,1250.0,40.0,CANCELLED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z,2460.0
ORD0007,2026-10-13,C007,P007,Product 8,GROCERY,MUMBAI,3,1425.0,60.0,PLACED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z,4215.0
ORD0008,2026-10-14,C008,P008,Product 1,BEVERAGES,KOCHI,4,200.0,0.0,DELIVERED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z,800.0
ORD0009,2026-10-15,C009,P009,Product 2,HOUSEHOLD,DELHI,5,375.0,20.0,CANCELLED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z,1855.0
ORD0010,2026-10-16,C010,P010,Product 3,GROCERY,COIMBATORE,1,550.0,40.0,PLACED,batch_001,online_shopping_initial.csv,2026-11-23T06:57:26.552Z,510.0


In [0]:
actual_silver_count = silver_df.count()
print("Actual Silver Count:", actual_silver_count)
print("Expected Silver Count:",expected_silver_count)

if(actual_silver_count == expected_silver_count):
  print("Reconciliation PASS")
else:
  print("Reconciliation FAILED")

Actual Silver Count: 36
0
Reconciliation FAILED


In [0]:
silver_df.write.mode("overwrite").format("delta").saveAsTable("assesment_pipline.silver.silver_orders")